In [ ]:
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List
import random
import math
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
try:
    from timm.utils import ModelEmaV3
except ModuleNotFoundError:
    import copy

    class ModelEmaV3:
        def __init__(self, model, decay=0.9999):
            self.module = copy.deepcopy(model).eval()
            self.decay = decay
            for param in self.module.parameters():
                param.requires_grad_(False)

        @torch.no_grad()
        def update(self, model):
            ema_state = self.module.state_dict()
            model_state = model.state_dict()
            for key, value in ema_state.items():
                if value.dtype.is_floating_point:
                    value.mul_(self.decay).add_(model_state[key].detach(), alpha=1 - self.decay)
                else:
                    value.copy_(model_state[key])

        def state_dict(self):
            return self.module.state_dict()

        def load_state_dict(self, state_dict):
            self.module.load_state_dict(state_dict)
from tqdm import tqdm
import matplotlib.pyplot as plt
import torch.optim as optim
import numpy as np
import os

In [ ]:
class SinusoidalEmbeddings(nn.Module):
    def __init__(self, time_steps:int, embed_dim: int):
        super().__init__()
        position = torch.arange(time_steps).unsqueeze(dim=1).float()
        div = torch.exp(torch.arange(0, embed_dim, 2).float() * -(math.log(10000) / embed_dim))
        embeddings = torch.zeros(time_steps, embed_dim, requires_grad=False)
        embeddings[:, 0::2] = torch.sin(position * div)
        embeddings[:, 1::2] = torch.cos(position * div)
        self.register_buffer('embeddings', embeddings)

    def forward(self, x, t):
        t = torch.as_tensor(t, dtype=torch.long, device=x.device)
        embeds = self.embeddings[t]
        return embeds[:, :, None, None]

In [ ]:
# Residual block
class ResBlock(nn.Module):
    def __init__(self, C: int, num_groups: int, dropout_prob: float):
        super().__init__()
        self.relu = nn.ReLU(inplace=True)
        self.gnorm1 = nn.GroupNorm(num_groups=num_groups, num_channels=C)
        self.gnorm2 = nn.GroupNorm(num_groups=num_groups, num_channels=C)
        self.conv1 = nn.Conv2d(C, C, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(C, C, kernel_size=3, padding=1)
        self.dropout = nn.Dropout(p=dropout_prob, inplace=True)

    def forward(self, x, embeddings):
        x = x + embeddings[:, :x.shape[1], :, :]
        r = self.conv1(self.relu(self.gnorm1(x)))
        r = self.dropout(r)
        r = self.conv2(self.relu(self.gnorm2(r)))
        return r + x

In [ ]:
class Attention(nn.Module):
    def __init__(self, C:int, num_heads:int, dropout_prob: float):
        super().__init__()
        self.proj1 = nn.Linear(C, C*3)
        self.proj2 = nn.Linear(C, C)
        self.num_heads = num_heads
        self.dropout_prob = dropout_prob

    def forward(self, x):
        b, c, h, w = x.shape
        x = x.permute(0, 2, 3, 1).reshape(b, h * w, c)
        x = self.proj1(x)
        x = x.reshape(b, h * w, 3, self.num_heads, c // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = x[0], x[1], x[2]
        x = F.scaled_dot_product_attention(q, k, v, is_causal=False, dropout_p=self.dropout_prob)
        x = x.permute(0, 2, 1, 3).reshape(b, h, w, c)
        x = self.proj2(x)
        return x.permute(0, 3, 1, 2)

In [ ]:
class UnetLayer(nn.Module):
    def __init__(self,
                 upscale: bool,
                 attention: bool,
                 num_groups: int,
                 dropout_prob: float,
                 num_heads: int,
                 C: int):
        super().__init__()
        self.ResBlock1 = ResBlock(C, num_groups, dropout_prob)
        self.ResBlock2 = ResBlock(C, num_groups, dropout_prob)
        if upscale:
            self.conv = nn.ConvTranspose2d(C, C//2, kernel_size=4, stride=2, padding=1)
        else:
            self.conv = nn.Conv2d(C, C*2, kernel_size=3, stride=2, padding=1)
        if attention:
            self.attention_layer = Attention(C, num_heads, dropout_prob)

    def forward(self, x, embeddings):
        x = self.ResBlock1(x, embeddings)
        if hasattr(self, 'attention_layer'):
            x = self.attention_layer(x)
        x = self.ResBlock2(x, embeddings)
        return self.conv(x), x

In [ ]:
class UNET(nn.Module):
    def __init__(self,
                 Channels: List = [64, 128, 256, 512, 512, 384],
                 Attentions: List = [False, True, False, False, False, True],
                 Upscales: List = [False, False, False, True, True, True],
                 num_groups: int = 32,
                 dropout_prob: float = 0.1,
                 num_heads: int = 8,
                 input_channels: int = 1,
                 output_channels: int = 1,
                 time_steps: int = 1000,
                 num_classes: int = 10):
        super().__init__()
        self.num_layers = len(Channels)
        self.num_classes = num_classes
        self.shallow_conv = nn.Conv2d(input_channels, Channels[0], kernel_size=3, padding=1)
        out_channels = (Channels[-1]//2) + Channels[0]
        self.late_conv = nn.Conv2d(out_channels, out_channels//2, kernel_size=3, padding=1)
        self.output_conv = nn.Conv2d(out_channels//2, output_channels, kernel_size=1)
        self.relu = nn.ReLU(inplace=True)
        embed_dim = max(Channels)
        self.embeddings = SinusoidalEmbeddings(time_steps=time_steps, embed_dim=embed_dim)
        # +1 cho null token (lop "khong dieu kien" dung cho classifier-free guidance)
        self.label_embeddings = nn.Embedding(num_classes + 1, embed_dim)
        for i in range(self.num_layers):
            layer = UnetLayer(
                upscale=Upscales[i],
                attention=Attentions[i],
                num_groups=num_groups,
                dropout_prob=dropout_prob,
                C=Channels[i],
                num_heads=num_heads
            )
            setattr(self, f'Layer{i+1}', layer)

    def forward(self, x, t, labels):
        x = self.shallow_conv(x)
        t_emb = self.embeddings(x, t)                          # (b, E, 1, 1)
        l_emb = self.label_embeddings(labels)[:, :, None, None]  # (b, E, 1, 1)
        cond = t_emb + l_emb                                   # gop time + label
        residuals = []
        for i in range(self.num_layers//2):
            layer = getattr(self, f'Layer{i+1}')
            x, r = layer(x, cond)
            residuals.append(r)
        for i in range(self.num_layers//2, self.num_layers):
            layer = getattr(self, f'Layer{i+1}')
            x = torch.concat((layer(x, cond)[0], residuals[self.num_layers-i-1]), dim=1)
        return self.output_conv(self.relu(self.late_conv(x)))

In [ ]:
class DDPM_Scheduler(nn.Module):
    def __init__(self, num_time_steps: int=1000):
        super().__init__()
        beta = torch.linspace(1e-4, 0.02, num_time_steps, requires_grad=False)
        self.register_buffer('beta', beta)
        alpha = 1 - self.beta
        # luu y: self.alpha la alpha_bar (cumprod), khong phai alpha tung buoc
        self.register_buffer('alpha', torch.cumprod(alpha, dim=0).requires_grad_(False))

    def forward(self, t):
        return self.beta[t], self.alpha[t]

In [ ]:
def set_seed(seed: int = 42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)

In [ ]:
def train(batch_size: int=64,
          num_time_steps: int=1000,
          num_epochs: int=40,
          seed: int=-1,
          ema_decay: float=0.999,
          lr=2e-4,
          num_classes: int=10,
          p_uncond: float=0.1,
          checkpoint_path: str=None):
    """Train UNET co dieu kien theo nhan, co label-dropout cho classifier-free guidance."""
    set_seed(random.randint(0, 2**32-1)) if seed == -1 else set_seed(seed)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=4)

    scheduler = DDPM_Scheduler(num_time_steps=num_time_steps).to(device)
    model = UNET(time_steps=num_time_steps, num_classes=num_classes).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    ema = ModelEmaV3(model, decay=ema_decay)
    if checkpoint_path is not None:
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['weights'])
        ema.load_state_dict(checkpoint['ema'])
        optimizer.load_state_dict(checkpoint['optimizer'])
    criterion = nn.MSELoss(reduction='mean')
    null_token = num_classes  # index cua lop "khong dieu kien"

    # Train song song tren nhieu GPU neu co (DataParallel chia batch cho cac GPU)
    if torch.cuda.device_count() > 1:
        print(f'Su dung {torch.cuda.device_count()} GPU voi DataParallel')
        train_model = nn.DataParallel(model)
    else:
        train_model = model

    for i in range(num_epochs):
        total_loss = 0
        for bidx, (x, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {i+1}/{num_epochs}")):
            x = x.to(device)
            labels = labels.to(device)
            # Classifier-free guidance: ngau nhien bo nhan -> null token
            drop = torch.rand(labels.shape[0], device=device) < p_uncond
            labels = torch.where(drop, torch.full_like(labels, null_token), labels)

            x = F.pad(x, (2, 2, 2, 2))
            t = torch.randint(0, num_time_steps, (batch_size,), device=device)
            e = torch.randn_like(x, requires_grad=False)
            a = scheduler.alpha[t].view(batch_size, 1, 1, 1)
            x = (torch.sqrt(a) * x) + (torch.sqrt(1 - a) * e)
            output = train_model(x, t, labels)
            optimizer.zero_grad()
            loss = criterion(output, e)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            ema.update(model)  # cap nhat tu model goc (DataParallel dung chung tham so)
        print(f'Epoch {i+1} | Loss {total_loss / (60000/batch_size):.5f}')

    checkpoint = {
        'weights': model.state_dict(),  # luu model goc -> key sach, load lai binh thuong
        'optimizer': optimizer.state_dict(),
        'ema': ema.state_dict()
    }
    os.makedirs('checkpoints', exist_ok=True)
    torch.save(checkpoint, 'checkpoints/cfg_checkpoint')

In [ ]:
def ddim_sample_cfg(checkpoint_path: str=None,
                    num_time_steps: int=1000,
                    ddim_steps: int=50,
                    eta: float=0.0,
                    guidance_scale: float=3.0,
                    num_classes: int=10,
                    samples_per_class: int=8,
                    ema_decay: float=0.999,
                    output_dir: str='outputs',
                    save_individual: bool=False):
    """Sinh anh co dieu kien bang DDIM + classifier-free guidance.

    - guidance_scale (w): 0 -> khong guidance; cang lon anh cang "dung lop" nhung
      it da dang hon. Thuong 2~5.
    - eta: 0 -> DDIM tat dinh; 1 -> DDPM stochastic.
    - output_dir: thu muc luu anh (luoi tong + tung anh rieng).
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model = UNET(time_steps=num_time_steps, num_classes=num_classes).to(device)
    model.load_state_dict(checkpoint['weights'])
    ema = ModelEmaV3(model, decay=ema_decay)
    ema.load_state_dict(checkpoint['ema'])
    scheduler = DDPM_Scheduler(num_time_steps=num_time_steps).to(device)
    null_token = num_classes

    os.makedirs(output_dir, exist_ok=True)

    # tap con timestep, di tu cao -> thap
    step_indices = torch.linspace(0, num_time_steps - 1, ddim_steps, device=device).long()
    step_indices = torch.flip(step_indices, dims=[0])

    model = ema.module.eval()
    fig, axes = plt.subplots(num_classes, samples_per_class,
                             figsize=(samples_per_class, num_classes))
    with torch.no_grad():
        for cls in range(num_classes):
            n = samples_per_class
            z = torch.randn(n, 1, 32, 32, device=device)
            labels = torch.full((n,), cls, dtype=torch.long, device=device)
            null_labels = torch.full((n,), null_token, dtype=torch.long, device=device)

            for i in range(ddim_steps):
                t = int(step_indices[i].item())
                t_tensor = torch.full((n,), t, dtype=torch.long, device=device)
                alpha_bar_t = scheduler.alpha[t]

                eps_cond = model(z, t_tensor, labels)
                eps_uncond = model(z, t_tensor, null_labels)
                # classifier-free guidance
                eps = eps_uncond + guidance_scale * (eps_cond - eps_uncond)

                x0 = (z - torch.sqrt(1 - alpha_bar_t) * eps) / torch.sqrt(alpha_bar_t)

                if i < ddim_steps - 1:
                    t_prev = int(step_indices[i + 1].item())
                    alpha_bar_prev = scheduler.alpha[t_prev]
                else:
                    alpha_bar_prev = torch.ones_like(alpha_bar_t)

                sigma = eta * torch.sqrt((1 - alpha_bar_prev) / (1 - alpha_bar_t)) \
                            * torch.sqrt(1 - alpha_bar_t / alpha_bar_prev)
                noise = torch.randn_like(z) if eta > 0 else torch.zeros_like(z)
                dir_xt = torch.sqrt(torch.clamp(1 - alpha_bar_prev - sigma**2, min=0.0)) * eps
                z = torch.sqrt(alpha_bar_prev) * x0 + dir_xt + sigma * noise

            imgs = z.clamp(0, 1).detach().cpu()
            for j in range(n):
                ax = axes[cls, j] if samples_per_class > 1 else axes[cls]
                arr = imgs[j].squeeze(0).numpy()
                ax.imshow(arr, cmap='gray')
                ax.axis('off')
                # luu tung anh rieng: outputs/class3_sample0.png
                if save_individual:
                    plt.imsave(os.path.join(output_dir, f'class{cls}_sample{j}.png'),
                               arr, cmap='gray')
    plt.tight_layout()
    # luu luoi tong hop
    grid_path = os.path.join(output_dir, f'grid_w{guidance_scale}_steps{ddim_steps}_eta{eta}.png')
    fig.savefig(grid_path, dpi=150, bbox_inches='tight')
    print(f'Da luu anh vao "{output_dir}/" (luoi: {grid_path})')
    plt.show()

In [ ]:
def main():
    train(lr=2e-4, num_epochs=40, ema_decay=0.999)
    # DDIM + classifier-free guidance, sinh 8 mau cho moi chu so 0-9
    ddim_sample_cfg('checkpoints/cfg_checkpoint',
                    ddim_steps=50, eta=0.0, guidance_scale=3.0,
                    samples_per_class=8)

if __name__ == '__main__':
    main()